#### create skewed dataset

In [0]:
from pyspark.sql import *
import random
spark=SparkSession.builder.appName("SkewDemo").getOrCreate()
data = []
for i in range (10000):
  # 85%skew hyderabad
  city = "Hyderabad" if random.random()<0.85 else random.choice(["Chennai","Mumbai","Delhi"])

  data.append((i,city,random.randint(100,10000)))
df_large = spark.createDataFrame(data,["id","city","amount"])
df_large.display()

### Dataset 2 (Small Table)

In [0]:
city_data = [
    ("Hyderabad", "South"),
    ("Chennai", "South"),
    ("Mumbai", "West"),
    ("Delhi", "North")
]

df_small = spark.createDataFrame(city_data, ["city", "region"])
df_small.display()

### IDENTIFY SKEW

In [0]:
df_large.groupBy("city").count().display()

In [0]:
joined_df = df_large.join(df_small, "city")
joined_df.count()

In [0]:
df_large.groupBy("city").count().display()

### SOLUTION 1: ADAPTIVE QUERY EXECUTION (AQE)

In [0]:
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")

In [0]:

joined_df = df_large.join(df_small, "city")
joined_df.count()

##### AQE does:
##### Detects skewed partition
##### Splits it into smaller chunks
##### Spark automatically fixes skew at runtime
##### Works only in Spark 3+

### SOLUTION 2: SALTING (Manual Fix)

In [0]:
from pyspark.sql.functions import rand

df_large_salt = df_large.withColumn("salt", (rand()*10).cast("int"))
df_large_salt.count()

#### Step 2: Expand small table

In [0]:
from pyspark.sql.functions import explode, array, lit

df_small_expanded = df_small.withColumn(
    "salt",
    explode(array([lit(i) for i in range(10)]))
)
df_small_expanded.count()

#### Join using salt

In [0]:
joined_salt = df_large_salt.join(df_small_expanded, ["city", "salt"])
joined_salt.count()

#### Hyderabad → split into 10 partitions
#### Balanced workload

### SOLUTION 3: BROADCAST JOIN (BEST)

In [0]:
from pyspark.sql.functions import broadcast

joined_broadcast = df_large.join(broadcast(df_small), "city")
joined_broadcast.count()

##### Small table sent to all executors
##### No shuffle
##### No skew impact

##### Avoid shuffle → avoid skew problem


In [0]:
# Broadcast join avoids shuffle entirely - no partitioning issues
print("Broadcast Join Result - No skew because no shuffle:")
joined_broadcast.groupBy("city").count().display()

In [0]:
# Demonstrate skew removal: Hyderabad split across 10 salt values
print("Original skewed distribution:")
df_large.groupBy("city").count().display()

print("\nAfter salting - Hyderabad distributed across 10 partitions:")
joined_salt.groupBy("city", "salt").count().orderBy("city", "salt").display()

In [0]:
# Comprehensive comparison of all three approaches
print("=" * 80)
print("PROBLEM: Original Skewed Data")
print("=" * 80)
print("Hyderabad has 85% of data → Single partition gets overloaded during shuffle\n")
df_large.groupBy("city").count().orderBy("city").display()

print("\n" + "=" * 80)
print("SOLUTION 1: AQE (Adaptive Query Execution)")
print("=" * 80)
print("✓ Spark automatically detects and splits skewed partitions at runtime")
print("✓ Works in Spark 3+, minimal code changes")
print("✓ Best for dynamic/unknown skew patterns\n")

print("\n" + "=" * 80)
print("SOLUTION 2: Salting (Manual Partition Distribution)")
print("=" * 80)
print("✓ Hyderabad's 8485 records split into 10 balanced partitions (~850 each)")
print("✓ Workload distributed across multiple executors")
print("✓ Best for known skew keys\n")
joined_salt.groupBy("city", "salt").count().orderBy("city", "salt").display()

print("\n" + "=" * 80)
print("SOLUTION 3: Broadcast Join (Eliminate Shuffle)")
print("=" * 80)
print("✓ Small table (4 rows) sent to ALL executors")
print("✓ NO SHUFFLE → NO SKEW PROBLEM")
print("✓ Fastest solution when small table < 10MB")
print("✓ Each executor processes data locally\n")
joined_broadcast.groupBy("city").count().orderBy("city").display()

print("\n" + "=" * 80)
print("KEY INSIGHT")
print("=" * 80)
print("• Salting: Distributes skew across partitions (still shuffles data)")
print("• Broadcast: Avoids shuffle entirely (copies small table to each executor)")
print("• AQE: Automatically applies both strategies as needed")